# Halden Energy — Week 6: Capital Allocation
**Python analysis notebook (parallel to the Excel workbook)**

Allocate a constrained budget across three projects. The budget funds roughly 1.5 of 3.

The twist: your cost of capital was set by your cohort's Week 4 transfer-pricing discipline. A disciplined cohort earns a low discount rate — and that changes which project is the best NPV choice, even though the IRR ranking never changes. Ranking by IRR funds the wrong portfolio when capital is cheap.

In [ ]:
import pandas as pd
cf    = pd.read_csv('data/project_cashflows.csv')
sched = pd.read_csv('data/cohort_discount_schedule.csv')
haircuts = pd.read_csv('data/forecast_haircuts.csv')

def npv(rate, flows): return sum(f/(1+rate)**t for t,f in enumerate(flows))
def irr(flows):
    lo,hi=-0.9,3.0
    for _ in range(300):
        m=(lo+hi)/2
        if npv(m,flows)>0: lo=m
        else: hi=m
    return m

projects = {'baton_rouge':cf['baton_rouge'].tolist(),
            'rotterdam':  cf['rotterdam'].tolist(),
            'helix':      cf['helix'].tolist()}
cf

## Part 1 — IRR of each project

IRR doesn't depend on the discount rate. Note the ordering: Baton Rouge highest, Helix lowest.

In [ ]:
for name, flows in projects.items():
    print(f'{name:12s} IRR={irr(flows)*100:5.1f}%  outlay={-flows[0]:4d}  undiscounted total={sum(flows):5d}')
# Baton Rouge 19.4%, Rotterdam 15.0%, Helix 12.4% — Helix lowest IRR but highest total return

## Part 2 — the cohort discount schedule, and how it flips the NPV ranking

Your Week 4 cohort discipline set your discount rate. Watch the NPV ranking change across the three cohort states.

In [ ]:
sched

In [ ]:
for _, row in sched.iterrows():
    r = row['discount_rate']
    npvs = {n: npv(r, f) for n, f in projects.items()}
    rank = ' > '.join(sorted(npvs, key=lambda k:-npvs[k]))
    print(f"{row['cohort_wk4_behavior']:11s} r={r:.3f} envelope=${row['capital_envelope_musd']}M | "
          f"BR={npvs['baton_rouge']:6.1f} Rot={npvs['rotterdam']:6.1f} Helix={npvs['helix']:6.1f} | {rank}")
print()
print('At the disciplined rate (6.5%), Helix has the HIGHEST NPV — but its IRR is the LOWEST.')
print('A team ranking by IRR funds Baton Rouge + Rotterdam and leaves the best NPV on the table.')

In [ ]:
# The crossover: below this discount rate Helix beats Baton Rouge on NPV; above it, BR wins.
lo,hi=0.0,0.20
for _ in range(200):
    m=(lo+hi)/2
    if npv(m,projects['helix'])>npv(m,projects['baton_rouge']): lo=m
    else: hi=m
print(f'BR/Helix NPV crossover: {m*100:.2f}%')
print(f"  disciplined 6.5% -> below crossover -> Helix wins on NPV")
print(f"  base 8.5% and lax 11.0% -> above crossover -> Baton Rouge wins on NPV")

## Part 3 — forecast-accuracy haircuts

The company's project forecasts have historically been optimistic, worst for new-business projects. Apply the haircuts and see the effect on Helix.

In [ ]:
haircuts

In [ ]:
# Map each project to its haircut: refining for BR/Rotterdam, new_business for Helix
haircut = {r['project_type']: r['actual_pct_of_forecast'] for _,r in haircuts.iterrows()}
project_type = {'baton_rouge':'refining','rotterdam':'refining','helix':'new_business'}
r_disciplined = 0.065
print('NPV at 6.5%, before and after the historical-accuracy haircut:')
for name, flows in projects.items():
    raw = npv(r_disciplined, flows)
    hc  = haircut[project_type[name]]
    print(f'{name:12s} raw={raw:6.1f}  haircut={hc:.0%}  adjusted={raw*hc:6.1f}')
print()
print('The 61% new-business haircut bites Helix hardest — exactly where the strategic case is made.')

## Part 4 — your analysis

Fill in the `# TODO` cells for your own cohort state.

In [ ]:
# TODO: set your cohort's discount rate (6.5% disciplined / 8.5% base / 11.0% lax)
# your_rate = None

# TODO: compute NPV of each project at your rate, and each IRR
# for name, flows in projects.items():
#     print(name, npv(your_rate, flows), irr(flows))

# TODO: does the NPV ranking agree with the IRR ranking at your rate?
# TODO: apply the haircuts. Does the ranking change?
# TODO: allocate within your envelope. Which projects do you fund, and why?

## Your decision

- Which projects do you fund, within your cohort's envelope?
- If you ranked by IRR, which project did you leave unfunded — and was that right at your discount rate?
- How did the forecast haircut change your thinking about Helix?

Remember: the IRR ranking is the same for every cohort. What changes is the discount rate your cohort earned, and at a low enough rate the highest-IRR project is no longer the highest-NPV project. Your Week 4 discipline created the exact condition under which an IRR-only decision costs the most.